> see `evaluate_properties_extraction.py` for most recent code.

# Evaluate Property Extraction

> y (ground truth) vs y_hat (LLM)

In [3]:
import os
import re
import sys

import numpy as np
import pandas as pd

from rouge_score import rouge_scorer

from sacrebleu.metrics import CHRF
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# print(notebook_dir)
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from metrics import EvaluationMetric
from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

In [4]:
# pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_columns', 40)
# pd.set_option('display.max_rows', None)

## Load Data

temperature = 0 for more control

In [5]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
# data_path = os.path.join(base_data_path, 'extract_prediction_properties/future-chronicle2050_fpb_synthethic-v1/openai/gpt-oss-120b/results.csv')
data_path = os.path.join(base_data_path, 'extract_prediction_properties-v1/evaluation/synthetic-fpb-chronicle2050-yt-news-timebank-mf_climate/openai_gpt-oss-120b/extracted_properties.csv')
df = DataProcessing.load_from_file(data_path)
df

,Input_Index,Sentence,Raw Response,Model Name,No Property,Source,Target,Date,Outcome,Dataset Source
0,0,"As the yen soared in recent years, Sansui's de...","{0: [""As"", ""the"", ""yen"", ""surged"", ""in"", ""rece...",openai/gpt-oss-120b,As|the|yen|surged|in|recent|years|Sansui's|dee...,NaN,NaN,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...
1,1,Waxman sells a variety of hardware products fo...,"{0: [""sells"", ""a"", ""variety"", ""of"", ""hardware""...",openai/gpt-oss-120b,sells|a|variety|of|hardware|products|for|the,Waxman,home repair market,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...
2,2,"Glaston , headquartered in Tampere , Finland ,...","{0: [""Glaston"", "","", ""headquartered"", ""in"", ""T...",openai/gpt-oss-120b,"Glaston|,|headquartered|in|Tampere|,|Finland|,...",NaN,NaN,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...
3,3,The overall weather trend will lead to a moder...,"{\n ""0"": [""The"", ""will"", ""lead"", ""to"", ""a"", ""...",openai/gpt-oss-120b,The|will|lead|to|a|by,NaN,overall weather trend,next Sunday,moderation of temperatures toward normal,synthetic-fpb-chronicle2050-yt-news-timebank-m...
4,4,The forecast indicates good predictability in ...,"{0: [""The"", ""forecast"", ""indicates"", ""in"", ""ba...",openai/gpt-oss-120b,The|forecast|indicates|in|based|on|guidance|fr...,GFS|ECMWF,mid to large-scale flow evolution,NaN,good predictability,synthetic-fpb-chronicle2050-yt-news-timebank-m...
5,5,StatesWest had proposed acquiring Mesa for $7 ...,"{0: [""had"", ""proposed"", ""acquiring"", ""for"", ""a...",openai/gpt-oss-120b,had|proposed|acquiring|for|and|one|share|of|a|...,StatesWest,Mesa,NaN,$7 a share|$3 a share,synthetic-fpb-chronicle2050-yt-news-timebank-m...
6,6,"But Kenneth Leon, a telecommunications analyst...","{0: [""But"", ""a"", ""telecommunications"", ""analys...",openai/gpt-oss-120b,But|a|telecommunications|analyst|with|Bear|Ste...,Kenneth Leon,BellSouth,five years,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...


In [6]:
col_names = df.iloc[: , 4:9]
col_names

,No Property,Source,Target,Date,Outcome
0,As|the|yen|surged|in|recent|years|Sansui's|dee...,NaN,NaN,NaN,NaN
1,sells|a|variety|of|hardware|products|for|the,Waxman,home repair market,NaN,NaN
2,"Glaston|,|headquartered|in|Tampere|,|Finland|,...",NaN,NaN,NaN,NaN
3,The|will|lead|to|a|by,NaN,overall weather trend,next Sunday,moderation of temperatures toward normal
4,The|forecast|indicates|in|based|on|guidance|fr...,GFS|ECMWF,mid to large-scale flow evolution,NaN,good predictability
5,had|proposed|acquiring|for|and|one|share|of|a|...,StatesWest,Mesa,NaN,$7 a share|$3 a share
6,But|a|telecommunications|analyst|with|Bear|Ste...,Kenneth Leon,BellSouth,five years,NaN


In [7]:
y_df = df.copy()
y_hat_df = df.copy()

In [ ]:
y_data_path = os.path.join(base_data_path, 'extraction_results/naacl_2026_submission/ground_truth/zero-shot/seed3/gpt-oss-120b/extracted_properties.csv') # COT GPT
y_hat_data_path = os.path.join(base_data_path, 'extraction_results/naacl_2026_submission/classification/zero-shot/seed3/gemma-3-27b-it/extracted_properties.csv')  # Zero-shot GPT
y_hat_data_path = os.path.join(base_data_path, 'extraction_results/naacl_2026_submission/classification/zero-shot/seed3/gemma-3-27b-it/extracted_properties.csv') # FEW-shot GPT

y_hat_data_path = os.path.join(base_data_path, 'extraction_results/naacl_2026_submission/classification/zero-shot/seed3/gemma-3-27b-it/extracted_properties.csv')
y_hat_data_path = os.path.join(base_data_path, 'extraction_results/naacl_2026_submission/classification/zero-shot/seed3/gemma-3-27b-it/extracted_properties.csv')
y_hat_data_path = os.path.join(base_data_path, 'extraction_results/naacl_2026_submission/classification/zero-shot/seed3/gemma-3-27b-it/extracted_properties.csv')


In [9]:
y_df = DataProcessing.load_from_file(y_data_path)
y_df = y_df.copy()
y_df.head(3)

,Seed,Input_Index,Base Sentence,Raw Response,Model Name,Parse Status,Source,Target,Date,Outcome,Dataset Name
0,3,0,"Five states in India, with nearly 300 million ...","{""1"": [], ""2"": [], ""3"": [""this month""], ""4"": []}",gpt-oss-120b,OK,NaN,NaN,this month,NaN,chronicle2050
1,3,1,"In addition, the directors said if a bid by Mr...","{""1"": [""the directors""], ""2"": [""Rank Organisat...",gpt-oss-120b,OK,the directors,Rank Organisation PLC,NaN,acquire the 51% of Cineplex's Film House unit ...,timebank
2,3,2,The poverty rate at the urban areas increased ...,"{""1"": [""Director of the Census Bureau""], ""2"": ...",gpt-oss-120b,OK,Director of the Census Bureau,NaN,January 2026,increased,synthetic


In [10]:
y_hat_df = DataProcessing.load_from_file(y_hat_data_path)
y_hat_df = y_hat_df.copy()
y_hat_df.head(3)

,Seed,Input_Index,Base Sentence,Raw Response,Model Name,Parse Status,Source,Target,Date,Outcome,Dataset Name
0,3,0,"Five states in India, with nearly 300 million ...","```json\n{""1"": [], ""2"": [""Five states in India...",gemma-3-27b-it,OK,NaN,Five states in India|India,this month,a broader sense of India’s political direction...,chronicle2050
1,3,1,"In addition, the directors said if a bid by Mr...","```json\n{""1"": [""the directors"", ""Cineplex""], ...",gemma-3-27b-it,OK,the directors|Cineplex,Mr. Drabinsky|Rank Organisation PLC|Cineplex's...,NaN,acquire the 51% of Cineplex's Film House unit ...,timebank
2,3,2,The poverty rate at the urban areas increased ...,"```json\n{""1"": [""Director of the Census Bureau...",gemma-3-27b-it,OK,Director of the Census Bureau,poverty rate at the urban areas,January 2026,increased,synthetic


In [11]:
col_names = y_df.columns.to_list()
col_names = col_names[6 : 10]
col_names

['Source', 'Target', 'Date', 'Outcome']

### Need to Consider

> All cases below represent scenarios where semantic meaning is preserved but surface-level differences cause cosine similarity to underperform or fail entirely.

---

1. **Standardizing Dates**
   - **Dates appear in many formats:** `January 1st, 2037`, `2037-01-01`, `01/01/2037`
   - **Consider:** Normalize to a canonical format (e.g., ISO 8601) before embedding or comparison to prevent penalizing semantically identical dates with different representations.
   - Dr. Grant: Look for libraries/models for data parsing; spaCy date parser

2. **Word Order**
   - **same set, different order:** `2037, 2038` vs `2038, 2037`
   - **subset vs full set:** `2037, 2038` vs `2037`
   - **Consider:** Set-based matching (e.g., Jaccard similarity) for multi-value fields alongside cosine similarity.

3. **Word Tense / Morphology**
   - **plural vs singular:** `Long Bets` vs `Long Bet`
   -  **Consider:** Lemmatization before embedding (spaCy supports this) would normalize these cases and more relevant for Target and Outcome fields. now the affects it'll have on actual sentence.

4. **Missing or Extra Characters**
   - **unit dropped:** `2.0C or higher` vs `2.0 or higher`
   - **spacing artifact:** `U.S. Departmentof Energy` vs `U.S. Department of Energy`
   -  **Consider:** Lightweight string normalization (lowercase, strip punctuation, fix spacing) as a preprocessing step before embedding. Know the affects it'll have on actual sentence.

## Embed Sentences

In [12]:
property_results = []

for col_name in col_names:
    print(f"Embeddings for {col_name}")
    y_spacy_fe = SpacyFeatureExtraction(y_df, col_name)
    embed_y_df = y_spacy_fe.sentence_embeddings_extraction(attach_to_df=True)

    y_hat_spacy_fe = SpacyFeatureExtraction(y_hat_df, col_name)
    embed_y_hat_df = y_hat_spacy_fe.sentence_embeddings_extraction(attach_to_df=True)

    np.testing.assert_equal(len(embed_y_df), len(embed_y_hat_df))

    property_result = {
        'property_name': col_name,
        'y_data': embed_y_df,
        'y_hat_data': embed_y_hat_df
    }
    property_results.append(property_result)

Embeddings for Source
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


100%|██████████| 2619/2619 [00:01<00:00, 1393.06it/s]


Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


100%|██████████| 2619/2619 [00:02<00:00, 993.46it/s] 


Embeddings for Target
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


...

100%|██████████| 2619/2619 [00:03<00:00, 843.03it/s]


Embeddings for Outcome
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


100%|██████████| 2619/2619 [00:07<00:00, 346.79it/s]


Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


100%|██████████| 2619/2619 [00:05<00:00, 439.91it/s]


In [13]:
property_results[0]['y_data']

,Seed,Input_Index,Base Sentence,Raw Response,Model Name,Parse Status,Source,Target,Date,Outcome,Dataset Name,Source Embedding,Target Embedding,Date Embedding,Outcome Embedding
0,3,0,"Five states in India, with nearly 300 million ...","{""1"": [], ""2"": [], ""3"": [""this month""], ""4"": []}",gpt-oss-120b,OK,NaN,NaN,this month,NaN,chronicle2050,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.1033025, 0.35113, 0.174804, 0.219125, -0.2...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,3,1,"In addition, the directors said if a bid by Mr...","{""1"": [""the directors""], ""2"": [""Rank Organisat...",gpt-oss-120b,OK,the directors,Rank Organisation PLC,NaN,acquire the 51% of Cineplex's Film House unit ...,timebank,"[0.218465, -0.35977, 0.12489, -0.2733225, 0.06...","[-0.14173633, 0.008530001, 0.19021334, -0.1547...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.02095351, 0.2105571, -0.0110858055, -0.049..."
2,3,2,The poverty rate at the urban areas increased ...,"{""1"": [""Director of the Census Bureau""], ""2"": ...",gpt-oss-120b,OK,Director of the Census Bureau,NaN,January 2026,increased,synthetic,"[-0.1445528, -0.0584426, -0.011615205, -0.2266...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.36162502, 0.271745, 0.3141255, 0.0701095, -...","[-0.68954, 0.74284, 0.19422, 0.05327, -0.34596..."
3,3,3,"In 2029-10-01, Research advisor Dr. Rodriguez ...","{""1"": [""Research advisor Dr. Rodriguez""], ""2"":...",gpt-oss-120b,OK,Research advisor Dr. Rodriguez,clinical trials at Pfizer,2029-10-01,success rate|some probability to remain stable,synthetic,"[-0.12546375, 0.1827738, 0.2209177, 0.24322, 0...","[-0.180035, 0.17989475, 0.034058005, -0.077822...","[-0.29055262, 0.520524, 0.0659592, 0.16700439,...","[-0.086083174, 0.23516166, -0.25333834, 0.0246..."
4,3,4,Nadia Popovici kept shifting her eyes from the...,"{""1"": [], ""2"": [""Nadia Popovici"", ""Brian Hamil...",gpt-oss-120b,OK,NaN,Nadia Popovici|Brian Hamilton,NaN,NaN,chronicle2050,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.05165967, 0.19298667, 0.14955999, -0.00669...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2614,3,2614,I'm going to say the Knicks grabbed the one seed.,"{""1"": [""I'm""], ""2"": [], ""3"": [], ""4"": [""grabbe...",gpt-oss-120b,OK,I'm,NaN,NaN,grabbed the one seed,yt,"[0.14912, 0.411815, -0.51933, -0.2507325, 0.09...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.000551749, -0.13124426, -0.03544375, 0.219..."
2615,3,2615,@ By Urban C. Lehner @ Staff Reporter of The W...,"{""1"": [""Urban C. Lehner"", ""Staff Reporter of T...",gpt-oss-120b,OK,Urban C. Lehner|Staff Reporter of The Wall Str...,"J.SNE|HOME ELECTRONICS, FURNISHINGS, APPLIANCE...",10/30/89,NaN,timebank,"[0.14276978, 0.11296199, -0.11552355, -0.08205...","[0.114684224, 0.24700972, 0.059623335, -0.0940...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2616,3,2616,"In Q4 2026, David envisions that the research ...","{""1"": [""David""], ""2"": [""the research and devel...",gpt-oss-120b,OK,David,the research and development expenses at Alpha...,Q4 2026,high probability to remain stable,synthetic,"[-0.57735, 0.53347, 0.25926, -0.38308, 0.12183...","[-0.15415001, 0.09393208, -0.0112869, -0.15746...","[0.13029501, 0.32928, 0.1604055, 0.20823951, -...","[-0.20645404, 0.29483, -0.28812423, 0.06771799..."
2617,3,2617,"Meteorologist John Lee predicts on 08/22/2028,...","{""1"": [""Meteorologist John Lee""], ""2"": [""the h...",gpt-oss-120b,OK,Meteorologist John Lee,the humidity at Miami,08/22/2028,may rise,synthetic,"[-0.27441666, 0.35177135, 0.13400434, -0.07901...","[-0.039136022, 0.38369247, 0.252195, -0.059116...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...

In [14]:
property_results[0]['y_hat_data']

,Seed,Input_Index,Base Sentence,Raw Response,Model Name,Parse Status,Source,Target,Date,Outcome,Dataset Name,Source Embedding,Target Embedding,Date Embedding,Outcome Embedding
0,3,0,"Five states in India, with nearly 300 million ...","```json\n{""1"": [], ""2"": [""Five states in India...",gemma-3-27b-it,OK,NaN,Five states in India|India,this month,a broader sense of India’s political direction...,chronicle2050,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.010007501, 0.051038425, 0.0825045, -0.16740...","[-0.1033025, 0.35113, 0.174804, 0.219125, -0.2...","[-0.07346161, 0.1416177, 0.1035859, 0.1038241,..."
1,3,1,"In addition, the directors said if a bid by Mr...","```json\n{""1"": [""the directors"", ""Cineplex""], ...",gemma-3-27b-it,OK,the directors|Cineplex,Mr. Drabinsky|Rank Organisation PLC|Cineplex's...,NaN,acquire the 51% of Cineplex's Film House unit ...,timebank,"[0.13602, -0.031015, -0.0942, 0.0116125, -0.00...","[0.08916187, 0.20009875, 0.18570426, -0.001716...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.02095351, 0.2105571, -0.0110858055, -0.049..."
2,3,2,The poverty rate at the urban areas increased ...,"```json\n{""1"": [""Director of the Census Bureau...",gemma-3-27b-it,OK,Director of the Census Bureau,poverty rate at the urban areas,January 2026,increased,synthetic,"[-0.1445528, -0.0584426, -0.011615205, -0.2266...","[-0.12095501, 0.262259, -0.037503663, -0.18573...","[0.36162502, 0.271745, 0.3141255, 0.0701095, -...","[-0.68954, 0.74284, 0.19422, 0.05327, -0.34596..."
3,3,3,"In 2029-10-01, Research advisor Dr. Rodriguez ...","```json\n{""1"": [""Research advisor Dr. Rodrigue...",gemma-3-27b-it,OK,Research advisor Dr. Rodriguez,Pfizer,2029-10-01,success rate of clinical trials|remain stable,synthetic,"[-0.12546375, 0.1827738, 0.2209177, 0.24322, 0...","[-0.42349, 0.045408, 0.12521, -0.5331, 0.04818...","[-0.29055262, 0.520524, 0.0659592, 0.16700439,...","[-0.26153967, 0.3300267, -0.1386605, -0.158955..."
4,3,4,Nadia Popovici kept shifting her eyes from the...,"```json\n{""1"": [], ""2"": [""Nadia Popovici"", ""Br...",gemma-3-27b-it,OK,NaN,Nadia Popovici|Brian Hamilton,NaN,NaN,chronicle2050,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.05165967, 0.19298667, 0.14955999, -0.00669...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2614,3,2614,I'm going to say the Knicks grabbed the one seed.,"```json\n{""1"": [""I""], ""2"": [""Knicks""], ""3"": []...",gemma-3-27b-it,OK,I,Knicks,NaN,the one seed,yt,"[0.18733, 0.40595, -0.51174, -0.55482, 0.03971...","[-0.61305, -0.045703, -0.033003, -0.42996, 0.6...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.029023672, 0.04400767, -0.07863434, 0.08223..."
2615,3,2615,@ By Urban C. Lehner @ Staff Reporter of The W...,"```json\n{""1"": [""@ By Urban C. Lehner @ Staff ...",gemma-3-27b-it,OK,@ By Urban C. Lehner @ Staff Reporter of The W...,NaN,10/30/89,NaN,timebank,"[0.0026125403, 0.1999662, -0.10681777, -0.0818...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2616,3,2616,"In Q4 2026, David envisions that the research ...","```json\n{""1"": [""David""], ""2"": [""Alphabet Inc....",gemma-3-27b-it,OK,David,Alphabet Inc.,Q4 2026,research and development expenses|remain stable,synthetic,"[-0.57735, 0.53347, 0.25926, -0.38308, 0.12183...","[0.00037500262, 0.110874996, -0.032168005, -0....","[0.13029501, 0.32928, 0.1604055, 0.20823951, -...","[-0.19202402, 0.02910932, -0.094951846, -0.168..."
2617,3,2617,"Meteorologist John Lee predicts on 08/22/2028,...","```json\n{""1"": [""Meteorologist John Lee""], ""2""...",gemma-3-27b-it,OK,Meteorologist John Lee,humidity at Miami,08/22/2028,may rise,synthetic,"[-0.27441666, 0.35177135, 0.13400434, -0.07901...","[-0.14286137, 0.5322667, 0.39906, -0.08656333,...","[0.0, 0.0, 0.

## Evaluation

In [19]:
# Initialize lexical metric objects once, outside the functions.
chrf_metric = CHRF(word_order=2)
rouge_metric = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)
bleu_smoothing = SmoothingFunction().method1


def is_missing_property(value):
    """
    Return True when a slot/property is absent.

    Handles None, NaN, empty strings, and strings containing only whitespace.
    """
    if value is None:
        return True

    if isinstance(value, float) and np.isnan(value):
        return True

    if isinstance(value, str) and value.strip() == "":
        return True

    return False


def normalize_property_text(value):
    """
    Normalize property text for strict matching.

    Lowercases, trims whitespace, collapses repeated spaces, and removes
    superficial spacing around pipe-delimited spans.
    """
    if is_missing_property(value):
        return ""

    text = str(value).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s*\|\s*', '|', text)

    return text


def compare_strict(y_word, y_hat_word):
    """
    Compare two extracted properties using normalized exact-string matching.

    Returns
    -------
    float
        1.0 if normalized strings match exactly; otherwise 0.0.
    """
    return float(
        normalize_property_text(y_word) == normalize_property_text(y_hat_word)
    )


def compare_chrf(y_word, y_hat_word):
    """
    Compare two extracted properties using character n-gram F-score (chrF++).

    Returns
    -------
    float
        chrF score on the 0-1 scale.
    """
    score = chrf_metric.sentence_score(
        hypothesis=str(y_hat_word),
        references=[str(y_word)]
    )

    return score.score / 100.0


def compare_bleu(y_word, y_hat_word):
    """
    Compare two extracted properties using sentence-level BLEU.

    Returns
    -------
    float
        BLEU score on the 0-1 scale.

    Notes
    -----
    BLEU is order-sensitive and can be harsh for short entity spans.
    Smoothing prevents zero scores for otherwise partially overlapping text.
    """
    reference_tokens = str(y_word).lower().split()
    hypothesis_tokens = str(y_hat_word).lower().split()

    if not reference_tokens or not hypothesis_tokens:
        return 0.0

    return sentence_bleu(
        [reference_tokens],
        hypothesis_tokens,
        smoothing_function=bleu_smoothing
    )


def compare_rouge(y_word, y_hat_word):
    """
    Compare two extracted properties using ROUGE-L F1.

    Returns
    -------
    float
        ROUGE-L F1 score on the 0-1 scale.
    """
    scores = rouge_metric.score(
        target=str(y_word),
        prediction=str(y_hat_word)
    )
    return scores['rougeL'].fmeasure


def compare_embeddings(y_word, y_hat_word, y_embed, y_hat_embed):
    """
    Compare two extracted properties using cosine similarity of embeddings.

    Returns
    -------
    float
        Cosine similarity, generally in [-1, 1].

    Notes
    -----
    If either vector is zero, fall back to strict string matching.
    """
    if y_embed is None or y_hat_embed is None:
        return 0.0

    y_embed = np.asarray(y_embed)
    y_hat_embed = np.asarray(y_hat_embed)

    if np.linalg.norm(y_embed) == 0 or np.linalg.norm(y_hat_embed) == 0:
        return compare_strict(y_word, y_hat_word)

    return EvaluationMetric.get_cosine_similarity(
        y_embed.reshape(1, -1),
        y_hat_embed.reshape(1, -1),
        per_row=False,
        idx=0
    )


def compare_property(
    y_word,
    y_hat_word,
    metric='embedding',
    y_embed=None,
    y_hat_embed=None,
    threshold=0.9
):
    """
    Compare one ground-truth property against one LLM-predicted property.

    Presence logic
    --------------
    GT present + prediction matches threshold -> TP
    GT present + prediction absent              -> FN
    GT present + prediction mismatches          -> FN
    GT absent  + prediction present             -> FP
    GT absent  + prediction absent              -> TN

    Parameters
    ----------
    y_word : str, None, or NaN
        Ground-truth property value.
    y_hat_word : str, None, or NaN
        LLM-predicted property value.
    metric : str
        One of: 'strict', 'chrf', 'bleu', 'rouge', or 'embedding'.
    y_embed : np.ndarray, optional
        Ground-truth embedding; required only for metric='embedding'.
    y_hat_embed : np.ndarray, optional
        Prediction embedding; required only for metric='embedding'.
    threshold : float
        Minimum metric score required for a semantic/lexical match.

    Returns
    -------
    dict
        A standardized row containing the evaluation category, score,
        binary ground-truth label, and binary predicted label.
    """
    y_present = not is_missing_property(y_word)
    y_hat_present = not is_missing_property(y_hat_word)

    result = {
        'y_word': y_word,
        'y_hat_word': y_hat_word,
        'metric': metric,
        'threshold': threshold,
        'score': None,
        'evaluation': None,
        'y': None,
        'y_hat': None
    }

    # Ground truth is empty; model correctly returns nothing.
    if not y_present and not y_hat_present:
        result.update({
            'evaluation': 'TN',
            'y': 0,
            'y_hat': 0
        })
        return result

    # Ground truth is empty; model hallucinates a property.
    if not y_present and y_hat_present:
        result.update({
            'evaluation': 'FP',
            'y': 0,
            'y_hat': 1
        })
        return result

    # Ground truth contains a property; model misses it entirely.
    if y_present and not y_hat_present:
        result.update({
            'evaluation': 'FN',
            'y': 1,
            'y_hat': 0
        })
        return result

    # Both ground truth and model prediction contain extracted text.
    if metric == 'strict':
        score = compare_strict(y_word, y_hat_word)

    elif metric == 'chrf':
        score = compare_chrf(y_word, y_hat_word)

    elif metric == 'bleu':
        score = compare_bleu(y_word, y_hat_word)

    elif metric == 'rouge':
        score = compare_rouge(y_word, y_hat_word)

    elif metric == 'embedding':
        score = compare_embeddings(
            y_word=y_word,
            y_hat_word=y_hat_word,
            y_embed=y_embed,
            y_hat_embed=y_hat_embed
        )

    else:
        raise ValueError(
            f"Unknown metric: '{metric}'. "
            "Choose from: strict, chrf, bleu, rouge, embedding."
        )

    result['score'] = score

    # Correct/sufficiently similar property extraction.
    if score >= threshold:
        result.update({
            'evaluation': 'TP',
            'y': 1,
            'y_hat': 1
        })

    # Simplified convention: incorrect span is an FN because the model
    # did not recover the correct gold property.
    else:
        result.update({
            'evaluation': 'FN',
            'y': 1,
            'y_hat': 0
        })

    return result


def map_words_to_labels(
    y_df,
    y_hat_df,
    col_name,
    metric='embedding',
    threshold=0.9
):
    """
    Evaluate one property column across all aligned rows.

    Parameters
    ----------
    y_df : pd.DataFrame
        Ground-truth property DataFrame.
    y_hat_df : pd.DataFrame
        Model property-prediction DataFrame.
    col_name : str
        Property column, such as 'Source', 'Target', 'Date', or 'Outcome'.
    metric : str
        Comparison metric: strict, chrf, bleu, rouge, or embedding.
    threshold : float
        Match threshold for the selected metric.

    Returns
    -------
    tuple
        tps, fns, fps, tns, comparison_df
    """
    if len(y_df) != len(y_hat_df):
        raise ValueError(
            f"DataFrames must have equal length. "
            f"Ground truth: {len(y_df)}, predictions: {len(y_hat_df)}"
        )

    tps = []
    fns = []
    fps = []
    tns = []
    comparisons = []

    embedding_col = f'{col_name} Embedding'

    for idx in range(len(y_df)):
        y_word = y_df[col_name].iloc[idx]
        y_hat_word = y_hat_df[col_name].iloc[idx]

        # Only retrieve embeddings when semantic comparison is selected.
        if metric == 'embedding':
            y_embed = y_df[embedding_col].iloc[idx]
            y_hat_embed = y_hat_df[embedding_col].iloc[idx]
        else:
            y_embed = None
            y_hat_embed = None

        comparison = compare_property(
            y_word=y_word,
            y_hat_word=y_hat_word,
            metric=metric,
            y_embed=y_embed,
            y_hat_embed=y_hat_embed,
            threshold=threshold
        )

        comparison['row_index'] = idx
        comparison['property'] = col_name
        comparisons.append(comparison)

        if comparison['evaluation'] == 'TP':
            tps.append(comparison)
        elif comparison['evaluation'] == 'FN':
            fns.append(comparison)
        elif comparison['evaluation'] == 'FP':
            fps.append(comparison)
        elif comparison['evaluation'] == 'TN':
            tns.append(comparison)

    comparison_df = pd.DataFrame(comparisons)

    return tps, fns, fps, tns, comparison_df

In [24]:
metrics = ['strict', 'chrf', 'bleu', 'rouge', 'embedding']

all_comparisons = {}

for metric in metrics:
    tps, fns, fps, tns, comparison_df = map_words_to_labels(
        y_df=y_df,
        y_hat_df=y_hat_df,
        col_name=col_name,
        metric=metric,
        threshold=0.9
    )
    all_comparisons[metric] = comparison_df
    print(f"\n{'='*40}")
    print(f"Metric: {metric.upper()}")
    print(f"{'='*40}")
    print(f"TP: {len(tps)} | FN: {len(fns)} | FP: {len(fps)} | TN: {len(tns)}")
    print(comparison_df[['row_index', 'y_word', 'y_hat_word', 'score', 'evaluation']].head(10))


Metric: STRICT
TP: 522 | FN: 1709 | FP: 191 | TN: 197
   row_index                                             y_word  \
0          0                                                NaN   
1          1  acquire the 51% of Cineplex's Film House unit ...   
2          2                                          increased   
3          3     success rate|some probability to remain stable   
4          4                                                NaN   
5          5                                                NaN   
6          6  the incidence of heart disease in the elderly ...   
7          7  ban Russian oil imports|ban all refined oil pr...   
8          8                                                NaN   
9          9  only a relative handful of Japanese companies|...   

                                          y_hat_word  score evaluation  
0  a broader sense of India’s political direction...    NaN         FP  
1  acquire the 51% of Cineplex's Film House unit ...    1.0  

In [23]:
comparison_df = pd.DataFrame(comparison_df)
comparison_df

,0,1,2,3,4,5,6,7,8,9,...,2609,2610,2611,2612,2613,2614,2615,2616,2617,2618
0,{'y_word': 'acquire the 51% of Cineplex's Film...,"{'y_word': 'increased', 'y_hat_word': 'increas...",{'y_word': 'ban Russian oil imports|ban all re...,"{'y_word': 'rose by 8.4 %', 'y_hat_word': 'ros...","{'y_word': '$ 4.2 million', 'y_hat_word': '$ 4...","{'y_word': 'would fall', 'y_hat_word': 'would ...","{'y_word': 'killing her', 'y_hat_word': 'killi...",{'y_word': 'purchase more Mesa stock or make a...,"{'y_word': 'need yet another shot', 'y_hat_wor...",{'y_word': 'amplified/blocky pattern|ridging t...,...,None,None,None,None,None,None,None,None,None,None
1,{'y_word': 'success rate|some probability to r...,{'y_word': 'the incidence of heart disease in ...,{'y_word': 'only a relative handful of Japanes...,{'y_word': 'negative height anomalies in north...,{'y_word': 'deadliest fire in New York since 1...,"{'y_word': 'a vaccine have been developed', 'y...","{'y_word': 'turnover rate|remained stable', 'y...","{'y_word': 'annual death toll|exceeded 90', 'y...",{'y_word': 'controls about 23 % of the Russian...,{'y_word': 'population density in Tokyo will h...,...,None,None,None,None,None,None,None,None,None,None
2,"{'y_word': nan, 'y_hat_word': 'a broader sense...","{'y_word': nan, 'y_hat_word': 'contracting con...","{'y_word': nan, 'y_hat_word': 'investigate who...","{'y_word': nan, 'y_hat_word': 'quality educati...","{'y_word': nan, 'y_hat_word': 'alter the cours...","{'y_word': nan, 'y_hat_word': 'nightmare', 'me...","{'y_word': nan, 'y_hat_word': 'a bit of drama'...","{'y_word': nan, 'y_hat_word': 'holds up', 'met...","{'y_word': nan, 'y_hat_word': 'interfere in th...","{'y_word': nan, 'y_hat_word': 'more resources'...",...,None,None,None,None,None,None,None,None,None,None
3,"{'y_word': nan, 'y_hat_word': nan, 'metric': '...","{'y_word': nan, 'y_hat_word': nan, 'metric': '...","{'y_word': nan, 'y_hat_word': nan, 'metric': '...","{'y_word': nan, 'y_hat_word': nan, 'metric': '...","{'y_word': nan, 'y_hat_word': nan, 'metric': '...","{'y_word': nan, 'y_hat_word': nan, 'metric': '...","{'y_word': nan, 'y_hat_word': nan, 'metric': '...","{'y_word': nan, 'y_hat_word': nan, 'metric': '...","{'y_word': nan, 'y_hat_word': nan, 'metric': '...","{'y_word': nan, 'y_hat_word': nan, 'metric': '...",...,None,None,None,None,None,None,None,None,None,None
4,y_word,y_hat_word,metric,threshold,score,evaluation,y,y_hat,row_index,property,...,None,None,None,None,None,None,None,None,None,None


### Compute Metrics Per Property

Builds classification report and confusion matrix across Source, Target, Date, and Outcome.

In [25]:
seed = 3
model_name = 'openai/gpt-oss-120b'
metrics_to_run = ['strict', 'chrf', 'bleu', 'rouge', 'embedding']
metrics_summary = []

for property_results_idx in range(len(property_results)):
    property_result = property_results[property_results_idx]

    property_name = property_result['property_name']
    print(f"\n{'='*60}")
    print(f"Property: {property_name}")
    print(f"{'='*60}")

    y_df     = property_result['y_data']
    y_hat_df = property_result['y_hat_data']

    for metric in metrics_to_run:
        print(f"\n  Metric: {metric.upper()}")

        tps, fns, fps, tns, comparison_df = map_words_to_labels(
            y_df=y_df,
            y_hat_df=y_hat_df,
            col_name=property_name,
            metric=metric,
            threshold=0.9
        )
        print(f"\t#TP: {len(tps)} | #FN: {len(fns)} | #FP: {len(fps)} | #TN: {len(tns)}")

        # --------------------------------------------------------
        # Build eval dataframe and compute classification metrics
        # --------------------------------------------------------
        tps_df = pd.DataFrame(tps)
        fns_df = pd.DataFrame(fns)
        fps_df = pd.DataFrame(fps)
        tns_df = pd.DataFrame(tns)

        eval_report_df   = DataProcessing.concat_dfs([tps_df, fns_df, fps_df, tns_df])
        actual_labels    = eval_report_df['y']
        predicted_labels = eval_report_df['y_hat']

        eval_report = EvaluationMetric.eval_classification_report(
            actual_labels, predicted_labels
        )

        confusion_mat, tn, fp, fn, tp = EvaluationMetric.get_confusion_matrix(
            actual_labels, predicted_labels, by_category=True
        )
        print(f"\tConfusion Matrix:\n{confusion_mat}\n")

        # --------------------------------------------------------
        # Build metrics row — one row per property per metric
        # --------------------------------------------------------
        metrics_row = {
            'seed':              seed,
            'model':             model_name,
            'property':          property_name,
            'metric':            metric,
            'test_accuracy':     eval_report.get('accuracy', None),
            'precision_class_0': eval_report.get('0', {}).get('precision', None),
            'precision_class_1': eval_report.get('1', {}).get('precision', None),
            'recall_class_0':    eval_report.get('0', {}).get('recall', None),
            'recall_class_1':    eval_report.get('1', {}).get('recall', None),
            'f1_class_0':        eval_report.get('0', {}).get('f1-score', None),
            'f1_class_1':        eval_report.get('1', {}).get('f1-score', None),
            'tn': tn,
            'fp': fp,
            'fn': fn,
            'tp': tp
        }
        metrics_summary.append(metrics_row)

metrics_summary_df = pd.DataFrame(metrics_summary)
print("\n" + "="*60)
print("METRICS SUMMARY")
print("="*60)
print(metrics_summary_df)


Property: Source

  Metric: STRICT
	#TP: 680 | #FN: 402 | #FP: 429 | #TN: 1108
              precision    recall  f1-score   support

           0       0.73      0.72      0.73      1537
           1       0.61      0.63      0.62      1082

    accuracy                           0.68      2619
   macro avg       0.67      0.67      0.67      2619
weighted avg       0.68      0.68      0.68      2619

	Confusion Matrix:
[[1108  429]
 [ 402  680]]


  Metric: CHRF
	#TP: 709 | #FN: 373 | #FP: 429 | #TN: 1108
              precision    recall  f1-score   support

           0       0.75      0.72      0.73      1537
           1       0.62      0.66      0.64      1082

    accuracy                           0.69      2619
   macro avg       0.69      0.69      0.69      2619
weighted avg       0.70      0.69      0.69      2619

	Confusion Matrix:
[[1108  429]
 [ 373  709]]


  Metric: BLEU
	#TP: 123 | #FN: 959 | #FP: 429 | #TN: 1108
              precision    recall  f1-score   suppor

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)
/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)


	#TP: 793 | #FN: 289 | #FP: 429 | #TN: 1108
              precision    recall  f1-score   support

           0       0.79      0.72      0.76      1537
           1       0.65      0.73      0.69      1082

    accuracy                           0.73      2619
   macro avg       0.72      0.73      0.72      2619
weighted avg       0.73      0.73      0.73      2619

	Confusion Matrix:
[[1108  429]
 [ 289  793]]


Property: Target

  Metric: STRICT
	#TP: 638 | #FN: 912 | #FP: 990 | #TN: 79
              precision    recall  f1-score   support

           0       0.08      0.07      0.08      1069
           1       0.39      0.41      0.40      1550

    accuracy                           0.27      2619
   macro avg       0.24      0.24      0.24      2619
weighted avg       0.26      0.27      0.27      2619

	Confusion Matrix:
[[ 79 990]
 [912 638]]


  Metric: CHRF
	#TP: 708 | #FN: 842 | #FP: 990 | #TN: 79
              precision    recall  f1-score   support

           0       0.

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)
/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)


	#TP: 730 | #FN: 820 | #FP: 990 | #TN: 79
              precision    recall  f1-score   support

           0       0.09      0.07      0.08      1069
           1       0.42      0.47      0.45      1550

    accuracy                           0.31      2619
   macro avg       0.26      0.27      0.26      2619
weighted avg       0.29      0.31      0.30      2619

	Confusion Matrix:
[[ 79 990]
 [820 730]]


  Metric: EMBEDDING
	#TP: 874 | #FN: 676 | #FP: 990 | #TN: 79
              precision    recall  f1-score   support

           0       0.10      0.07      0.09      1069
           1       0.47      0.56      0.51      1550

    accuracy                           0.36      2619
   macro avg       0.29      0.32      0.30      2619
weighted avg       0.32      0.36      0.34      2619

	Confusion Matrix:
[[ 79 990]
 [676 874]]


Property: Date

  Metric: STRICT
	#TP: 1137 | #FN: 494 | #FP: 71 | #TN: 917
              precision    recall  f1-score   support

           0       0.65

...

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)
/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)


	#TP: 655 | #FN: 1576 | #FP: 191 | #TN: 197
              precision    recall  f1-score   support

           0       0.11      0.51      0.18       388
           1       0.77      0.29      0.43      2231

    accuracy                           0.33      2619
   macro avg       0.44      0.40      0.30      2619
weighted avg       0.68      0.33      0.39      2619

	Confusion Matrix:
[[ 197  191]
 [1576  655]]


  Metric: EMBEDDING


/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)


	#TP: 1142 | #FN: 1089 | #FP: 191 | #TN: 197
              precision    recall  f1-score   support

           0       0.15      0.51      0.24       388
           1       0.86      0.51      0.64      2231

    accuracy                           0.51      2619
   macro avg       0.50      0.51      0.44      2619
weighted avg       0.75      0.51      0.58      2619

	Confusion Matrix:
[[ 197  191]
 [1089 1142]]


METRICS SUMMARY
    seed                model property     metric  test_accuracy  \
0      3  openai/gpt-oss-120b   Source     strict       0.682703   
1      3  openai/gpt-oss-120b   Source       chrf       0.693776   
2      3  openai/gpt-oss-120b   Source       bleu       0.470027   
3      3  openai/gpt-oss-120b   Source      rouge       0.689576   
4      3  openai/gpt-oss-120b   Source  embedding       0.725850   
5      3  openai/gpt-oss-120b   Target     strict       0.273769   
6      3  openai/gpt-oss-120b   Target       chrf       0.300496   
7      3  openai/gpt

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)


In [26]:
eval_report_df

,y_word,y_hat_word,metric,threshold,score,evaluation,y,y_hat,row_index,property
0,acquire the 51% of Cineplex's Film House unit ...,acquire the 51% of Cineplex's Film House unit ...,embedding,0.9,1.0,TP,1,1,1,Outcome
1,increased,increased,embedding,0.9,1.0,TP,1,1,2,Outcome
2,ban Russian oil imports|ban all refined oil pr...,ban Russian oil imports|ban all refined oil pr...,embedding,0.9,1.0,TP,1,1,7,Outcome
3,rose by 8.4 %,rose by 8.4 %,embedding,0.9,1.0,TP,1,1,10,Outcome
4,$ 4.2 million,$ 4.2 million,embedding,0.9,1.0,TP,1,1,12,Outcome
...,...,...,...,...,...,...,...,...,...,...
2614,NaN,NaN,embedding,0.9,NaN,TN,0,0,2571,Outcome
2615,NaN,NaN,embedding,0.9,NaN,TN,0,0,2581,Outcome
2616,NaN,NaN,embedding,0.9,NaN,TN,0,0,2586,Outcome
2617,NaN,NaN,embedding,0.9,NaN,TN,0,0,2613,Outcome


In [27]:
metrics_summary_df

,seed,model,property,metric,test_accuracy,precision_class_0,precision_class_1,recall_class_0,recall_class_1,f1_class_0,f1_class_1,tn,fp,fn,tp
0,3,openai/gpt-oss-120b,Source,strict,0.682703,0.733775,0.613165,0.720885,0.628466,0.727273,0.620721,1108,429,402,680
1,3,openai/gpt-oss-120b,Source,chrf,0.693776,0.748143,0.623023,0.720885,0.655268,0.734261,0.638739,1108,429,373,709
2,3,openai/gpt-oss-120b,Source,bleu,0.470027,0.536043,0.222826,0.720885,0.113678,0.614872,0.150551,1108,429,959,123
3,3,openai/gpt-oss-120b,Source,rouge,0.689576,0.742627,0.619343,0.720885,0.645102,0.731595,0.631960,1108,429,384,698
4,3,openai/gpt-oss-120b,Source,embedding,0.725850,0.793128,0.648936,0.720885,0.732902,0.755283,0.688368,1108,429,289,793
5,3,openai/gpt-oss-120b,Target,strict,0.273769,0.079717,0.391892,0.073901,0.411613,0.076699,0.401510,79,990,912,638
6,3,openai/gpt-oss-120b,Target,chrf,0.300496,0.085776,0.416961,0.073901,0.456774,0.079397,0.435961,79,990,842,708
7,3,openai/gpt-oss-120b,Target,bleu,0.070256,0.051837,0.095890,0.073901,0.067742,0.060933,0.079395,79,990,1445,105
8,3,openai/gpt-oss-120b,Target,rouge,0.308897,0.087875,0.424419,0.073901,0.470968,0.080285,0.446483,79,990,820,730
9,3,openai/gpt-oss-120b,Target,embedding,0.363879,0.104636,0.468884,0.073901,0.563871,0.086623,0.512009,79,990,676,874


In [ ]:
# eval_save_path = os.path.join(base_data_path, 'classification_results', 'properties')

# DataProcessing.save_to_file(
#     metrics_summary_df, 
#     path=eval_save_path, 
#     prefix='metrics_summary_llms', 
#     save_file_type='csv', 
#     include_version=True,
#     )
# print(f"✓ Saved metrics summary to: {os.path.join(eval_save_path, 'metrics_summary_llms.csv')}")